In [8]:
import pandas as pd
import glob
import os

In [21]:
# Specify the path to your input and output CSV files
path = "/Users/E2074/local-datasets/customer/age-prediction/local_extract/final/"

# Define the appropriate column names
column_names = ["customer_id", "age_group", "confidence_tag", "run_date"]

In [31]:
# Get all parquet files in the folder
parquet_files = glob.glob(os.path.join(path, '*.csv'))
parquet_files = sorted(parquet_files)
parquet_files

['/Users/E2074/local-datasets/customer/age-prediction/local_extract/final/customer_predicted_age_apr_may_0.csv',
 '/Users/E2074/local-datasets/customer/age-prediction/local_extract/final/customer_predicted_age_apr_may_1.csv',
 '/Users/E2074/local-datasets/customer/age-prediction/local_extract/final/customer_predicted_age_apr_may_2.csv',
 '/Users/E2074/local-datasets/customer/age-prediction/local_extract/final/customer_predicted_age_apr_may_3.csv',
 '/Users/E2074/local-datasets/customer/age-prediction/local_extract/final/customer_predicted_age_apr_may_4.csv',
 '/Users/E2074/local-datasets/customer/age-prediction/local_extract/final/customer_predicted_age_apr_may_5.csv',
 '/Users/E2074/local-datasets/customer/age-prediction/local_extract/final/customer_predicted_age_feb_mar_0.csv',
 '/Users/E2074/local-datasets/customer/age-prediction/local_extract/final/customer_predicted_age_feb_mar_1.csv',
 '/Users/E2074/local-datasets/customer/age-prediction/local_extract/final/customer_predicted_age

In [33]:
# Read and concatenate all parquet files into a single DataFrame
df = pd.concat([pd.read_csv(file) for file in parquet_files], ignore_index=True)

In [34]:
df.shape

(73894114, 4)

In [35]:
df.customer_id.nunique()

62779697

In [40]:
print(df[df['confidence_tag'] == 'VERY_HIGH'].shape)
print(df[df['confidence_tag'] == 'VERY_HIGH'].customer_id.nunique())

(27682159, 4)
17896444


In [42]:
df.confidence_tag.unique()

array(['VERY_HIGH', 'MEDIUM', 'HIGH', 'LOW', 'UNKNOWN'], dtype=object)

In [43]:
# Convert run_date to datetime
df['run_date'] = pd.to_datetime(df['run_date'])

# Define confidence tag order
confidence_order = {'VERY_HIGH': 0, 'HIGH': 1, 'MEDIUM': 2, 'LOW': 3, 'UNKNOWN' : 4}
df['confidence_rank'] = df['confidence_tag'].map(confidence_order)

# Sort by customer_id, confidence_rank, and run_date descending
df_sorted = df.sort_values(by=['customer_id', 'confidence_rank', 'run_date'], ascending=[True, True, False])

# Drop duplicates keeping the best row for each customer
df_unique = df_sorted.drop_duplicates(subset='customer_id', keep='first').drop(columns='confidence_rank')

In [44]:
df_unique.shape

(62779697, 4)

In [45]:
df_unique.customer_id.nunique()

62779697

In [46]:
df_unique

,customer_id,age_group,confidence_tag,run_date
10825694,5737c6acddbec2203f73315e,36-45,HIGH,2025-05-17
33138280,5737c6adddbec2203f73316a,21-25,HIGH,2025-02-10
41559412,5737c6adddbec2203f73316d,26-30,VERY_HIGH,2025-02-10
41994075,5737c6aeddbec2203f733173,31-35,HIGH,2025-02-10
5492174,5737c6aeddbec2203f733176,36-45,VERY_HIGH,2025-05-17
...,...,...,...,...
2807047,6828d57a6fb6851dffca11c3,Above-45,HIGH,2025-05-17
9111142,6828d57d02b7a97ec0ec1754,36-45,LOW,2025-05-17
2984142,6828d57f02b7a914acec1831,Above-45,HIGH,2025-05-17
4318652,6828d5883b257e339a4c413e,36-45,LOW,2025-05-17


In [49]:
df_unique = df_unique.reset_index(drop=True)

In [50]:
df_unique

,customer_id,age_group,confidence_tag,run_date
0,5737c6acddbec2203f73315e,36-45,HIGH,2025-05-17
1,5737c6adddbec2203f73316a,21-25,HIGH,2025-02-10
2,5737c6adddbec2203f73316d,26-30,VERY_HIGH,2025-02-10
3,5737c6aeddbec2203f733173,31-35,HIGH,2025-02-10
4,5737c6aeddbec2203f733176,36-45,VERY_HIGH,2025-05-17
...,...,...,...,...
62779692,6828d57a6fb6851dffca11c3,Above-45,HIGH,2025-05-17
62779693,6828d57d02b7a97ec0ec1754,36-45,LOW,2025-05-17
62779694,6828d57f02b7a914acec1831,Above-45,HIGH,2025-05-17
62779695,6828d5883b257e339a4c413e,36-45,LOW,2025-05-17


In [53]:
df_unique.age_group.value_counts(normalize=True)

age_group
21-25       0.313170
36-45       0.258968
26-30       0.175294
31-35       0.146358
Above-45    0.075560
Below-21    0.024603
UNKNOWN     0.006047
Name: proportion, dtype: float64

In [55]:
df_unique.confidence_tag.value_counts(normalize=True)

confidence_tag
HIGH         0.360109
VERY_HIGH    0.285067
MEDIUM       0.281304
LOW          0.067473
UNKNOWN      0.006047
Name: proportion, dtype: float64

In [57]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [58]:
## Connection
connection = presto.connect(
        host='processing-2.processing.data.production.internal' ,
    # 'presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)


presto_port = '80'
username = 'manoj.ravirajan@rapido.bike'
conn1 = presto.connect('bi-presto.serving.data.production.internal', presto_port, username)
conn2 = presto.connect('bi-trino-2.serving.data.production.internal', presto_port, username)
conn3= presto.connect('processing-2.processing.data.production.internal',presto_port,username)
conn4= presto.connect('processing.processing.data.production.internal',presto_port,username)

In [59]:
## base

base_query = f"""
select 
    customerid customer_id,
    CASE 
    WHEN age < 21 THEN 'Below-21'
    WHEN age BETWEEN 21 AND 25 THEN '21-25'
    WHEN age BETWEEN 26 AND 30 THEN '26-30'
    WHEN age BETWEEN 31 AND 35 THEN '31-35'
    WHEN age BETWEEN 36 AND 45 THEN '36-45'
    WHEN age > 45 THEN 'Above-45' 
    ELSE 'UNKNOWN' END age_group,
    'VERY_HIGH' confidence_tag,
    '2025-05-17' run_date
from
    datasets.customer_single_view 
where 
    mobileOS = 'IOS'
    and dateOfBirth is not null
"""

In [60]:
df_base_query = pd.read_sql(base_query, conn3)
df_base_query

,customer_id,age_group,confidence_tag,run_date
0,588dfc4188dda8ef4529b6f6,31-35,VERY_HIGH,2025-05-17
1,590302aa25a655ed295e9c33,26-30,VERY_HIGH,2025-05-17
2,594cc11885992d305990cd5a,36-45,VERY_HIGH,2025-05-17
3,59f48f182980677d92faf279,31-35,VERY_HIGH,2025-05-17
4,5a100aaa66830b3cfb9f05a5,31-35,VERY_HIGH,2025-05-17
...,...,...,...,...
1728073,6144b2aee8f2238d1aa87ffa,Above-45,VERY_HIGH,2025-05-17
1728074,61457876d0796f55bb05c903,31-35,VERY_HIGH,2025-05-17
1728075,61458fbfe8f22323f8a90487,26-30,VERY_HIGH,2025-05-17
1728076,6145a46ede864c553600604c,21-25,VERY_HIGH,2025-05-17


In [ ]:
df_base_query 

In [61]:
df_unique

,customer_id,age_group,confidence_tag,run_date
0,5737c6acddbec2203f73315e,36-45,HIGH,2025-05-17
1,5737c6adddbec2203f73316a,21-25,HIGH,2025-02-10
2,5737c6adddbec2203f73316d,26-30,VERY_HIGH,2025-02-10
3,5737c6aeddbec2203f733173,31-35,HIGH,2025-02-10
4,5737c6aeddbec2203f733176,36-45,VERY_HIGH,2025-05-17
...,...,...,...,...
62779692,6828d57a6fb6851dffca11c3,Above-45,HIGH,2025-05-17
62779693,6828d57d02b7a97ec0ec1754,36-45,LOW,2025-05-17
62779694,6828d57f02b7a914acec1831,Above-45,HIGH,2025-05-17
62779695,6828d5883b257e339a4c413e,36-45,LOW,2025-05-17


In [62]:
# # Convert run_date to datetime
# df1['run_date'] = pd.to_datetime(df1['run_date'])
# df2['run_date'] = pd.to_datetime(df2['run_date'])

# Concatenate
df_combined = pd.concat([df_base_query, df_unique], ignore_index=True)

In [63]:
df_combined

,customer_id,age_group,confidence_tag,run_date
0,588dfc4188dda8ef4529b6f6,31-35,VERY_HIGH,2025-05-17
1,590302aa25a655ed295e9c33,26-30,VERY_HIGH,2025-05-17
2,594cc11885992d305990cd5a,36-45,VERY_HIGH,2025-05-17
3,59f48f182980677d92faf279,31-35,VERY_HIGH,2025-05-17
4,5a100aaa66830b3cfb9f05a5,31-35,VERY_HIGH,2025-05-17
...,...,...,...,...
64507770,6828d57a6fb6851dffca11c3,Above-45,HIGH,2025-05-17 00:00:00
64507771,6828d57d02b7a97ec0ec1754,36-45,LOW,2025-05-17 00:00:00
64507772,6828d57f02b7a914acec1831,Above-45,HIGH,2025-05-17 00:00:00
64507773,6828d5883b257e339a4c413e,36-45,LOW,2025-05-17 00:00:00


In [65]:
# Convert run_date to datetime
df_combined['run_date'] = pd.to_datetime(df_combined['run_date'])

In [66]:
# Define confidence tag order
confidence_order = {'VERY_HIGH': 0, 'HIGH': 1, 'MEDIUM': 2, 'LOW': 3, 'UNKNOWN' : 4}
df_combined['confidence_rank'] = df_combined['confidence_tag'].map(confidence_order)

In [67]:
df_combined

,customer_id,age_group,confidence_tag,run_date,confidence_rank
0,588dfc4188dda8ef4529b6f6,31-35,VERY_HIGH,2025-05-17,0
1,590302aa25a655ed295e9c33,26-30,VERY_HIGH,2025-05-17,0
2,594cc11885992d305990cd5a,36-45,VERY_HIGH,2025-05-17,0
3,59f48f182980677d92faf279,31-35,VERY_HIGH,2025-05-17,0
4,5a100aaa66830b3cfb9f05a5,31-35,VERY_HIGH,2025-05-17,0
...,...,...,...,...,...
64507770,6828d57a6fb6851dffca11c3,Above-45,HIGH,2025-05-17,1
64507771,6828d57d02b7a97ec0ec1754,36-45,LOW,2025-05-17,3
64507772,6828d57f02b7a914acec1831,Above-45,HIGH,2025-05-17,1
64507773,6828d5883b257e339a4c413e,36-45,LOW,2025-05-17,3


In [68]:
# Sort by customer_id, confidence_rank, and run_date descending
df_sorted = df_combined.sort_values(by=['customer_id', 'confidence_rank', 'run_date'], ascending=[True, True, False])

In [69]:
# Drop duplicates keeping the best row for each customer
df_final_age = df_sorted.drop_duplicates(subset='customer_id', keep='first').drop(columns='confidence_rank')

In [70]:
df_final_age.shape

(64218615, 4)

In [71]:
df_final_age.customer_id.nunique()

64218615

In [75]:
df_final_age.to_csv('/Users/E2074/local-datasets/customer/age-prediction/local_extract/final/age_dataset_may_2025.csv', index=False)

In [72]:
df_final_age.confidence_tag.value_counts(normalize=True)

confidence_tag
HIGH         0.351955
VERY_HIGH    0.301243
MEDIUM       0.274941
LOW          0.065951
UNKNOWN      0.005909
Name: proportion, dtype: float64

In [73]:
df_final_age.age_group.value_counts(normalize=True)

age_group
21-25       0.310999
36-45       0.256482
26-30       0.178240
31-35       0.147756
Above-45    0.075197
Below-21    0.025417
UNKNOWN     0.005910
Name: proportion, dtype: float64

In [74]:
df_final_age.confidence_tag.value_counts()

confidence_tag
HIGH         22602058
VERY_HIGH    19345424
MEDIUM       17656335
LOW           4235305
UNKNOWN        379493
Name: count, dtype: int64

In [76]:
pd.read_csv('/Users/E2074/local-datasets/customer/age-prediction/local_extract/final/age_dataset_may_2025.csv')

,customer_id,age_group,confidence_tag,run_date
0,5737c6acddbec2203f73315e,36-45,HIGH,2025-05-17
1,5737c6adddbec2203f73316a,21-25,HIGH,2025-02-10
2,5737c6adddbec2203f73316d,26-30,VERY_HIGH,2025-02-10
3,5737c6aeddbec2203f733173,31-35,HIGH,2025-02-10
4,5737c6aeddbec2203f733176,36-45,VERY_HIGH,2025-05-17
...,...,...,...,...
64218610,682a0d12d6195b03a62eb46a,36-45,VERY_HIGH,2025-05-17
64218611,682a0d54c6335d029f730623,21-25,VERY_HIGH,2025-05-17
64218612,682a1e1adc515e7c76bfd1c7,26-30,VERY_HIGH,2025-05-17
64218613,682a25a26fb68552c242b152,21-25,VERY_HIGH,2025-05-17
